# MGANet — PASCAL-5i on Google Colab

**Runtime → Change runtime type → Hardware accelerator = GPU** (T4 is fine; A100/H100/L4 for bf16).

Pipeline:
1. Locate the repo at `/content/MGANet`
2. Install deps
3. Download + extract VOC2012
4. Build PASCAL-5i lists + configs
5. Train / test


## 1. Locate repo

In [ ]:
import os, sys
REPO = "/content/MGANet"

# If the repo is not there yet, clone your copy (edit the URL) or upload it:
# !git clone <YOUR_REPO_URL> $REPO

if not os.path.isdir(REPO):
    raise SystemExit("Repo not found at %s — clone/upload it first." % REPO)
%cd $REPO
print("cwd:", os.getcwd())
assert os.path.isfile("train.py"), "train.py missing — wrong REPO path?"


## 2. Check GPU

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print("device:", name)
    print("bf16 capable:", any(k in name for k in ("A100", "H100", "L4", "L40", "A10")))


## 3. Install dependencies

Colab already ships torch/torchvision/numpy/opencv. This mainly installs `visdom` and friends.
Do **not** let pip replace the CUDA torch build.


In [ ]:
%cd /content/MGANet
!python -m pip install -q networkx scikit-image visdom tensorboardX tqdm PyYAML
import torch, cv2, sklearn, skimage, networkx, tensorboardX, visdom
print("deps OK")


## 3b. Download ImageNet backbone (required)

`model/resnet.py` expects `./pretrained_model/resnet50_v2.pth` (deep-base ResNet-50:
split `conv1/conv2/conv3` stem, **not** torchvision's 7x7 stem).

It comes from the semseg/PFENet `initmodel` Drive folder.
If `gdown` fails (Drive quota), download `resnet50_v2.pth` manually and upload it to
`/content/MGANet/pretrained_model/resnet50_v2.pth`.


In [ ]:
%cd /content/MGANet
import os, glob, shutil
os.makedirs("pretrained_model", exist_ok=True)
DST = "pretrained_model/resnet50_v2.pth"
if not os.path.isfile(DST):
    !python -m pip install -q gdown
    !gdown --folder https://drive.google.com/drive/folders/1Hrz1wOxOZm4nIIS7UMJeL79AQrdvpj6v -O /tmp/pfenet_initmodel
    hits = glob.glob("/tmp/pfenet_initmodel/**/resnet50_v2.pth", recursive=True)
    if not hits:
        raise SystemExit("resnet50_v2.pth not found; download it manually into pretrained_model/")
    shutil.copy(hits[0], DST)
print("backbone:", DST, os.path.getsize(DST), "bytes")


## 4. Download + extract PASCAL VOC2012 (train/val)

In [ ]:
%cd /content/MGANet
import os
if not os.path.isdir("data/VOC2012/JPEGImages"):
    !wget -q --show-progress http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar
    !mkdir -p data
    !tar -xf VOCtrainval_11-May-2012.tar -C data
    !mv data/VOCdevkit/VOC2012 data/VOC2012
    !rm -rf data/VOCdevkit VOCtrainval_11-May-2012.tar
else:
    print("VOC2012 already present")
!ls data/VOC2012 | head


## 5. Build PASCAL-5i lists + configs

Reads `data/splits/pascal/{trn,val}/fold*.txt` and writes
`lists/pascal/**` plus `config/pascal/pascal_split{0..3}_resnet50_MGANet.yaml`.

Split convention: split `f` → novel classes `5f+1..5f+5`, base = other 15.

The BAM/PASCAL-5i splits assume **VOC2012 + SBD** masks. Plain VOC2012 has
`SegmentationClass` for only ~2913 images, so SBD-only ids would crash the
Dataset (`cv2.imread` → `None` → empty `label_class`). `--drop_missing`
drops those pairs (and pairs whose mask lacks the listed class).
For the full split, also download SBD `cls` masks into
`data/VOC2012/SegmentationClass/` and rerun without the flag.


In [ ]:
%cd /content/MGANet
!python tools/prepare_pascal5i.py --voc_root data/VOC2012 --drop_missing


## 6. Sanity-check the generated lists

In [ ]:
%cd /content/MGANet
import os
for s in range(4):
    for mode in ("train", "val"):
        p = f"lists/pascal/fss_list/{mode}/data_list_{s}.txt"
        assert os.path.isfile(p), p
        with open(p) as f:
            n = sum(1 for _ in f)
        print(f"split {s} {mode}: {n} images")


## 7. Train

Edit `SPLIT`, `SHOT`, `BATCH`. AMP dtype is picked from the GPU:
- A100/H100/L4/L40/A10 → `bfloat16`
- T4/V100/P100 → `float16` (GradScaler)


In [ ]:
%cd /content/MGANet
import torch

SPLIT = 0        # 0..3
SHOT  = 1        # 1 or 5
BATCH = 4        # lower to 2 if OOM

name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
dtype = "bfloat16" if any(k in name for k in ("A100", "H100", "L4", "L40", "A10")) else "float16"
cmd = (f"python train.py "
       f"--config config/pascal/pascal_split{SPLIT}_resnet50_MGANet.yaml "
       f"--opts shot {SHOT} batch_size {BATCH} amp True amp_dtype {dtype}")
print(cmd)
!{cmd}


## 8. Test

Set `WEIGHT` to a trained checkpoint, e.g.
`exp/MGANet_iSAID/pascal/MGANet/split0/resnet50/snapshot/train_epoch_XXX_0.XXXX.pth`.
The proto dict path is in the config (`Test_Finetune.proto_dict_path`); update it if needed.


In [ ]:
%cd /content/MGANet
WEIGHT = ""   # <-- fill in
SPLIT = 0
if not WEIGHT:
    print("Set WEIGHT to a checkpoint before testing.")
else:
    cmd = (f"python test.py "
           f"--config config/pascal/pascal_split{SPLIT}_resnet50_MGANet.yaml "
           f"--opts weight {WEIGHT}")
    print(cmd)
    !{cmd}


## Notes

- Run everything from the repo root; list files use relative paths (`data/VOC2012/...`).
- PASCAL configs use `train_h = val_size = 473`, `resnet50`, `workers 2`.
- If Colab runs out of RAM, set `workers 0` via `--opts workers 0`.
- Visualizer always makes `./vis/LoveDA/...`; harmless, unused unless `--viz`.
- Checkpoints land in `exp/MGANet_iSAID/pascal/MGANet/split<k>/resnet50/snapshot/`.
- Save to Drive before the runtime dies:
  `from google.colab import drive; drive.mount('/content/drive')`
  then `cp -r exp /content/drive/MyDrive/MGANet_exp`.
